In [2]:
import pandas as pd
import numpy as np


In [3]:
sales = pd.read_csv("../data/raw/sales_transactions.csv")

In [4]:
sales.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
0,2025-04-02,RCPT00000001,ST16,SKU02498,CUST01410,1,2379.11,2379.11,In-Store,0.0,NaN
1,2025-04-02,RCPT00000001,ST16,SKU04596,CUST01410,3,335.55,1006.65,In-Store,0.0,NaN
2,2022-04-24,RCPT00000002,ST15,SKU00078,CUST00134,1,820.53,820.53,Online,0.0,NaN
3,2022-04-24,RCPT00000002,ST15,SKU00554,CUST00134,2,88.32,176.64,Online,0.0,NaN
4,2024-09-22,RCPT00000003,ST20,SKU03727,CUST08826,1,1660.93,1660.93,Online,0.0,NaN


In [5]:
sales.tail()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
9945506,2023-09-09,RCPT05181381,ST04,SKU00701,CUST06397,2,39.24,68.51,In-Store,12.7,PROMO010
9945507,2025-12-03,RCPT05181382,ST03,SKU02163,CUST04945,1,791.70,791.70,In-Store,0.0,NaN
9945508,2025-12-03,RCPT05181382,ST03,SKU02866,CUST04945,4,94.26,377.04,In-Store,0.0,NaN
9945509,2023-01-20,RCPT05181383,ST03,SKU03727,CUST05244,1,1660.93,1660.93,In-Store,0.0,NaN
9945510,2023-07-01,RCPT05181384,ST20,SKU04596,CUST08714,1,335.55,335.55,In-Store,0.0,NaN


In [6]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9945511 entries, 0 to 9945510
Data columns (total 11 columns):
 #   Column        Dtype  
---  ------        -----  
 0   date          object 
 1   receipt_id    object 
 2   store_id      object 
 3   sku_id        object 
 4   customer_id   object 
 5   quantity      int64  
 6   unit_price    float64
 7   total_value   float64
 8   channel       object 
 9   discount_pct  float64
 10  promo_id      object 
dtypes: float64(3), int64(1), object(7)
memory usage: 834.7+ MB


In [7]:
sales.describe()

,quantity,unit_price,total_value,discount_pct
count,9.945511e+06,9.945511e+06,9.945511e+06,9.945511e+06
mean,1.880336e+00,6.215749e+02,1.094512e+03,6.289965e+00
std,1.089063e+00,6.751952e+02,1.536814e+03,1.383267e+01
min,1.000000e+00,2.441000e+01,1.225000e+01,0.000000e+00
25%,1.000000e+00,1.734300e+02,2.351700e+02,0.000000e+00
50%,2.000000e+00,3.367200e+02,5.432000e+02,0.000000e+00
75%,3.000000e+00,9.256700e+02,1.277610e+03,0.000000e+00
max,5.000000e+00,4.755470e+03,2.377735e+04,4.980000e+01


In [8]:
sales.isnull().sum()

date                  0
receipt_id            0
store_id              0
sku_id                0
customer_id           0
quantity              0
unit_price            0
total_value           0
channel               0
discount_pct          0
promo_id        7855345
dtype: int64

In [9]:
sales.nunique()

date               1461
receipt_id      5167864
store_id             30
sku_id             5000
customer_id       10000
quantity              5
unit_price         4837
total_value      182745
channel               3
discount_pct         84
promo_id             96
dtype: int64

In [10]:
sales["channel"].value_counts(normalize=True)*100

channel
In-Store      55.269277
Online        29.743560
Mobile App    14.987164
Name: proportion, dtype: float64

In [11]:
sales["quantity"].value_counts()

quantity
1    4971513
2    2486694
3    1491849
4     696817
5     298638
Name: count, dtype: int64

In [12]:
sales["discount_pct"].value_counts().sort_index().head(15)

discount_pct
0.0     7855345
5.0          27
5.8        3695
5.9       11035
6.8        3994
7.0       67557
7.9        8646
8.1        2295
8.5       15013
8.6       16784
9.0       77458
9.7       13730
9.9       15984
10.2       6156
10.3       1880
Name: count, dtype: int64

In [13]:
# check business perspective
(sales["quantity"]*sales["unit_price"]==sales["total_value"]).sum()

np.int64(7375195)

In [14]:
# value doesnot match with 0.0 per discount
diff=sales["quantity"]*sales["unit_price"]-sales["total_value"]
diff.describe()

count    9.945511e+06
mean     7.431736e+01
std      2.953767e+02
min     -3.637979e-12
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.044346e+04
dtype: float64

In [15]:
diff.eq(0).sum()

np.int64(7375195)

In [16]:
diff.ne(0).sum()

np.int64(2570316)

In [17]:
mismatch=diff.ne(0)

sales.loc[mismatch,["quantity","unit_price","discount_pct","promo_id","total_value"]].head()

,quantity,unit_price,discount_pct,promo_id,total_value
1,3,335.55,0.0,NaN,1006.65
10,3,961.05,0.0,NaN,2883.15
15,2,233.64,49.7,PROMO030,235.04
19,1,1000.23,48.6,PROMO037,514.12
20,3,602.13,0.0,NaN,1806.39


In [18]:
diff.abs().describe()

count    9.945511e+06
mean     7.431736e+01
std      2.953767e+02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      5.684342e-14
max      1.044346e+04
dtype: float64

In [20]:
diff.abs().sort_values(ascending=True).head(10)

0          0.0
6137990    0.0
6137989    0.0
6137988    0.0
6137987    0.0
6137986    0.0
6137984    0.0
6137983    0.0
6137982    0.0
6137979    0.0
dtype: float64

In [23]:
(diff.abs().gt(1e-8).sum())

np.int64(2090166)

In [28]:
expected_total=(sales["quantity"]*sales["unit_price"]*(1-sales["discount_pct"]/100))
expected_total.head()

0    2379.11
1    1006.65
2     820.53
3     176.64
4    1660.93
dtype: float64

In [26]:
(expected_total-sales["total_value"]).abs().describe()

count    9.945511e+06
mean     5.281566e-04
std      1.219890e-03
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      5.684342e-14
max      5.000000e-03
dtype: float64

In [29]:
sales.duplicated().sum()

np.int64(12897)

In [33]:
sales[sales.duplicated(keep=False)].sort_values("receipt_id").head(20)

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
270,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN
272,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN
314,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN
315,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN
544,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN
545,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN
2935,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045
2936,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045
5845,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN
5846,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN


In [35]:
sales[sales.duplicated(keep=False)]["receipt_id"].nunique()

12739

In [36]:
sales[sales.duplicated(keep=False)].groupby(list(sales.columns)).size().value_counts().sort_index()

2    2656
3      30
Name: count, dtype: int64

In [38]:
dup_rows=sales[sales.duplicated(keep=False)]
dup_rows.groupby(list(sales.columns),dropna=False).size().value_counts().sort_index()

2    12598
3      148
4        1
Name: count, dtype: int64

In [39]:
sales["date"].min(),sales["date"].max()

('2022-01-01', '2025-12-31')

In [40]:
pd.to_datetime(sales["date"]).nunique()

1461

In [41]:
dates=pd.to_datetime(sales["date"])
dates.max()-dates.min()

Timedelta('1460 days 00:00:00')

In [42]:
pd.date_range(start=dates.min(),end=dates.max()).difference(dates)

DatetimeIndex([], dtype='datetime64[ns]', freq='D')

In [43]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9945511 entries, 0 to 9945510
Data columns (total 11 columns):
 #   Column        Dtype  
---  ------        -----  
 0   date          object 
 1   receipt_id    object 
 2   store_id      object 
 3   sku_id        object 
 4   customer_id   object 
 5   quantity      int64  
 6   unit_price    float64
 7   total_value   float64
 8   channel       object 
 9   discount_pct  float64
 10  promo_id      object 
dtypes: float64(3), int64(1), object(7)
memory usage: 834.7+ MB


In [44]:
sales["channel"].nunique()

3

In [45]:
sales["store_id"].nunique()

30

In [46]:
sales["sku_id"].nunique(),sales["customer_id"].nunique()

(5000, 10000)

In [51]:
sales[["quantity","unit_price","total_value","discount_pct"]].describe()

,quantity,unit_price,total_value,discount_pct
count,9.945511e+06,9.945511e+06,9.945511e+06,9.945511e+06
mean,1.880336e+00,6.215749e+02,1.094512e+03,6.289965e+00
std,1.089063e+00,6.751952e+02,1.536814e+03,1.383267e+01
min,1.000000e+00,2.441000e+01,1.225000e+01,0.000000e+00
25%,1.000000e+00,1.734300e+02,2.351700e+02,0.000000e+00
50%,2.000000e+00,3.367200e+02,5.432000e+02,0.000000e+00
75%,3.000000e+00,9.256700e+02,1.277610e+03,0.000000e+00
max,5.000000e+00,4.755470e+03,2.377735e+04,4.980000e+01


In [52]:
sales[["channel"]].value_counts()

channel   
In-Store      5496812
Online        2958149
Mobile App    1490550
Name: count, dtype: int64

In [53]:
sales[["store_id", "sku_id", "customer_id", "promo_id"]].nunique()

store_id          30
sku_id          5000
customer_id    10000
promo_id          96
dtype: int64

In [54]:
sales["customer_id"].unique()

array(['CUST01410', 'CUST00134', 'CUST08826', ..., 'CUST07767',
       'CUST07626', 'CUST09371'], shape=(10000,), dtype=object)

In [55]:
sales[sales.duplicated(keep=False)].sort_values(
    ["receipt_id", "date"]
).head(20)

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
270,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN
272,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN
314,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN
315,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN
544,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN
545,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN
2935,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045
2936,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045
5845,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN
5846,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN


In [56]:
sales_clean = sales.drop_duplicates()

sales.shape, sales_clean.shape

((9945511, 11), (9932614, 11))

In [57]:
sales_clean.duplicated().sum()

np.int64(0)

In [58]:
sales_clean["date"]=pd.to_datetime(sales_clean["date"])

C:\Users\aadi1\AppData\Local\Temp\ipykernel_19232\2190478579.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales_clean["date"]=pd.to_datetime(sales_clean["date"])


In [59]:
sales_clean["date"].dtype

dtype('<M8[ns]')

In [60]:
sales_clean=sales.drop_duplicates().copy()

In [61]:
sales_clean["date"] = pd.to_datetime(sales_clean["date"])

In [63]:
sales_clean.shape

(9932614, 11)

In [67]:
sales_clean.duplicated().sum()

np.int64(0)

In [68]:
sales.duplicated().sum()

np.int64(12897)

In [69]:
sales_clean.isna().sum()

date                  0
receipt_id            0
store_id              0
sku_id                0
customer_id           0
quantity              0
unit_price            0
total_value           0
channel               0
discount_pct          0
promo_id        7845164
dtype: int64

In [70]:
sales_clean.dtypes

date            datetime64[ns]
receipt_id              object
store_id                object
sku_id                  object
customer_id             object
quantity                 int64
unit_price             float64
total_value            float64
channel                 object
discount_pct           float64
promo_id                object
dtype: object

In [71]:
expected_total.describe()

count    9.945511e+06
mean     1.094512e+03
std      1.536814e+03
min      1.225382e+01
25%      2.351700e+02
50%      5.432000e+02
75%      1.277610e+03
max      2.377735e+04
dtype: float64

In [72]:
expected_value=(sales_clean["quantity"]*sales_clean["unit_price"]*(1-sales_clean["discount_pct"]/100))
(sales_clean["total_value"]-expected_value).abs().max()

0.005000000001018634